In [8]:
# Regularised Logistic Regression for Species Distribution
# Single-species vs Multi-species approach

import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

In [9]:
train = pd.read_csv("../data/raw/train.csv")

target_col = "pres.abs"
species_col = "Species"
id_col = "id"

print(train.columns)
print(train.shape)
print(train.head())

print("\nTarget distribution:")
print(train[target_col].value_counts())

print("\nSpecies counts:")
print(train[species_col].value_counts())

Index(['long', 'lat', 'disturb', 'rainann', 'soildepth', 'soilfert', 'tempann',
       'topo', 'easting', 'northing', 'plot', 'Species', 'pres.abs', 'id'],
      dtype='str')
(5128, 14)
       long      lat  disturb  rainann  soildepth  soilfert  tempann  topo  \
0  153.4889 -28.1669        3     1588       1139         2      200     0   
1  153.4889 -28.1669        3     1588       1139         2      200     0   
2  153.4889 -28.1669        3     1588       1139         2      200     0   
3  153.4889 -28.1669        3     1588       1139         2      200     0   
4  153.4889 -28.1669        3     1588       1139         2      200     0   

        easting      northing  plot                     Species  pres.abs  id  
0 -41493.928042  6.871991e+06     1            Cacophis kreftii         0   1  
1 -41493.928042  6.871991e+06     1     Calyptotis scutirostrum         0   2  
2 -41493.928042  6.871991e+06     1  Coeranoscincus reticulatus         0   3  
3 -41493.928042  6.871991

In [10]:
X = train.drop(columns=[target_col])
y = train[target_col]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True))
print("y_val distribution:")
print(y_val.value_counts(normalize=True))

X_train: (4102, 13)
X_val: (1026, 13)
y_train distribution:
pres.abs
0    0.936616
1    0.063384
Name: proportion, dtype: float64
y_val distribution:
pres.abs
0    0.936647
1    0.063353
Name: proportion, dtype: float64


In [11]:
X_train_multi = X_train.drop(columns=[id_col])
X_val_multi = X_val.drop(columns=[id_col])

numeric_features = X_train_multi.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train_multi.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

preprocessor_multi = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

multi_logreg_l2 = Pipeline(steps=[
    ("preprocessor", preprocessor_multi),
    ("model", LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced"
    ))
])

multi_logreg_l2.fit(X_train_multi, y_train)

y_pred_multi = multi_logreg_l2.predict(X_val_multi)
y_prob_multi = multi_logreg_l2.predict_proba(X_val_multi)[:, 1]

multi_l2_results = {
    "Model": "Multi-species Logistic Regression L2",
    "Accuracy": accuracy_score(y_val, y_pred_multi),
    "F1": f1_score(y_val, y_pred_multi),
    "AUC": roc_auc_score(y_val, y_prob_multi)
}

multi_l2_results

Numeric features: ['long', 'lat', 'disturb', 'rainann', 'soildepth', 'soilfert', 'tempann', 'topo', 'easting', 'northing', 'plot']
Categorical features: ['Species']


{'Model': 'Multi-species Logistic Regression L2',
 'Accuracy': 0.7417153996101364,
 'F1': 0.2857142857142857,
 'AUC': 0.8225886496438004}

In [12]:
# ============================================================
# 4. Multi-species Logistic Regression with L1
# ============================================================

multi_logreg_l1 = Pipeline(steps=[
    ("preprocessor", preprocessor_multi),
    ("model", LogisticRegression(
        penalty="l1",
        C=1.0,
        solver="liblinear",
        max_iter=1000,
        class_weight="balanced"
    ))
])

multi_logreg_l1.fit(X_train_multi, y_train)

y_pred_l1 = multi_logreg_l1.predict(X_val_multi)
y_prob_l1 = multi_logreg_l1.predict_proba(X_val_multi)[:, 1]

multi_l1_results = {
    "Model": "Multi-species Logistic Regression L1",
    "Accuracy": accuracy_score(y_val, y_pred_l1),
    "F1": f1_score(y_val, y_pred_l1),
    "AUC": roc_auc_score(y_val, y_prob_l1)
}

multi_l1_results

{'Model': 'Multi-species Logistic Regression L1',
 'Accuracy': 0.7426900584795322,
 'F1': 0.2864864864864865,
 'AUC': 0.8227327303289843}

In [13]:
# ============================================================
# 5. Single-species Logistic Regression
# ============================================================

single_species_results = []

species_list = train[species_col].unique()

for sp in species_list:
    print("\nRunning model for:", sp)
    
    train_sp = train[train[species_col] == sp].copy()
    
    X_sp = train_sp.drop(columns=[target_col])
    y_sp = train_sp[target_col]
    
    if y_sp.nunique() < 2:
        print(f"Skipping {sp}: only one class exists.")
        continue
    
    X_train_sp, X_val_sp, y_train_sp, y_val_sp = train_test_split(
        X_sp,
        y_sp,
        test_size=0.2,
        random_state=42,
        stratify=y_sp
    )
    
    X_train_sp = X_train_sp.drop(columns=[id_col, species_col])
    X_val_sp = X_val_sp.drop(columns=[id_col, species_col])
    
    numeric_features_sp = X_train_sp.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_features_sp = X_train_sp.select_dtypes(include=["object", "category"]).columns.tolist()
    
    preprocessor_sp = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features_sp),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features_sp)
        ]
    )
    
    model_sp = Pipeline(steps=[
        ("preprocessor", preprocessor_sp),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=1000,
            class_weight="balanced"
        ))
    ])
    
    model_sp.fit(X_train_sp, y_train_sp)
    
    y_pred_sp = model_sp.predict(X_val_sp)
    y_prob_sp = model_sp.predict_proba(X_val_sp)[:, 1]
    
    single_species_results.append({
        "Species": sp,
        "Model": "Single-species Logistic Regression L2",
        "Accuracy": accuracy_score(y_val_sp, y_pred_sp),
        "F1": f1_score(y_val_sp, y_pred_sp),
        "AUC": roc_auc_score(y_val_sp, y_prob_sp),
        "Train size": len(X_train_sp),
        "Validation size": len(X_val_sp),
        "Presence rate": y_sp.mean()
    })

single_results_df = pd.DataFrame(single_species_results)
single_results_df


Running model for: Cacophis kreftii

Running model for: Calyptotis scutirostrum

Running model for: Coeranoscincus reticulatus

Running model for: Egernia mcpheei

Running model for: Eulamprus murrayi

Running model for: Ophioscincus truncatus

Running model for: Pseudechis porphyricaus

Running model for: Saltuarius swaini


,Species,Model,Accuracy,F1,AUC,Train size,Validation size,Presence rate
0,Cacophis kreftii,Single-species Logistic Regression L2,0.821705,0.258065,0.900000,512,129,0.031201
1,Calyptotis scutirostrum,Single-species Logistic Regression L2,0.844961,0.500000,0.915966,512,129,0.078003
2,Coeranoscincus reticulatus,Single-species Logistic Regression L2,0.705426,0.472222,0.801587,512,129,0.162246
3,Egernia mcpheei,Single-species Logistic Regression L2,0.767442,0.062500,0.618110,512,129,0.018721
4,Eulamprus murrayi,Single-species Logistic Regression L2,0.922481,0.000000,0.320312,512,129,0.004680
5,Ophioscincus truncatus,Single-species Logistic Regression L2,0.666667,0.441558,0.748564,512,129,0.177847
6,Pseudechis porphyricaus,Single-species Logistic Regression L2,0.837209,0.000000,0.563492,512,129,0.021841
7,Saltuarius swaini,Single-species Logistic Regression L2,0.775194,0.000000,0.618110,512,129,0.012480


In [14]:
# ============================================================
# 6. Summarise single-species results
# ============================================================

single_summary = {
    "Model": "Single-species Logistic Regression L2 - Average",
    "Accuracy": single_results_df["Accuracy"].mean(),
    "F1": single_results_df["F1"].mean(),
    "AUC": single_results_df["AUC"].mean()
}

single_summary

{'Model': 'Single-species Logistic Regression L2 - Average',
 'Accuracy': np.float64(0.7926356589147288),
 'F1': np.float64(0.21679314748871198),
 'AUC': np.float64(0.6857678901402114)}

In [15]:
# ============================================================
# 7. Final comparison table for your model
# ============================================================

comparison_table = pd.DataFrame([
    multi_l2_results,
    multi_l1_results,
    single_summary
])

comparison_table

,Model,Accuracy,F1,AUC
0,Multi-species Logistic Regression L2,0.741715,0.285714,0.822589
1,Multi-species Logistic Regression L1,0.742690,0.286486,0.822733
2,Single-species Logistic Regression L2 - Average,0.792636,0.216793,0.685768


In [16]:
# Features and target
X_train = train.drop(columns=[target_col])
y_train = train[target_col]